# Step 0: Pre-trained E vs From-Scratch

**Claim**: Pre-trained encoder (E) from simulated BSs converges faster on unseen BS than training from scratch.

## Experiment Design
1. Pre-train E + θ_task on BS₁~₆ (centralized or FL)
2. Deploy to BS₇ (unseen): adapt θ_BS only (freeze E + θ_task)
3. Compare vs. training full model from scratch on BS₇
4. Plot: NMSE vs. training samples / epochs

In [ ]:
import sys, os
sys.path.insert(0, '/workspace')
os.chdir('/workspace')

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

from src.config import DatasetConfig, ModelConfig, TrainConfig
from src.data.dataset import ChannelEstimationDataset, PerBSDataLoader
from src.data.utils import nmse_db
from src.models.estimator import SiteAwareEstimator, create_model
from src.models.baselines import PlainEstimator
from src.training.trainer import train_local, evaluate

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# Config
DATA_DIR = 'data/channels'
PRETRAIN_BS = [0, 1, 2, 3, 4, 5]
TEST_BS = [6, 7]
SNR_RANGE = (0.0, 30.0)
BATCH_SIZE = 32
PRETRAIN_EPOCHS = 100
ADAPT_EPOCHS = 50

## Phase 1: Pre-train on BS₁~₆

In [ ]:
# Pre-training dataset (all pre-train BSs pooled)
pretrain_ds = ChannelEstimationDataset(
    data_dir=DATA_DIR, bs_ids=PRETRAIN_BS, snr_range_db=SNR_RANGE
)
n_total = len(pretrain_ds)
n_val = int(n_total * 0.15)
n_train = n_total - n_val
train_ds, val_ds = torch.utils.data.random_split(pretrain_ds, [n_train, n_val])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

print(f'Pre-train samples: {n_train} train, {n_val} val')

In [ ]:
# Pre-train 3-way model
pretrained_model = create_model(site_integration='film', site_embed_dim=64).to(device)
pretrain_result = train_local(
    pretrained_model, train_loader, val_loader,
    epochs=PRETRAIN_EPOCHS, lr=1e-3, device=device
)
print(f'Pre-train best val NMSE: {10*np.log10(pretrain_result["best_val"]):.2f} dB')

## Phase 2: Adapt to Unseen BS₇

In [ ]:
# Test BS dataset
test_bs_id = TEST_BS[0]  # BS₇
test_ds = ChannelEstimationDataset(
    data_dir=DATA_DIR, bs_ids=[test_bs_id], snr_range_db=SNR_RANGE
)
n_test_total = len(test_ds)
n_test_val = int(n_test_total * 0.3)
n_test_train = n_test_total - n_test_val
test_train_ds, test_val_ds = torch.utils.data.random_split(test_ds, [n_test_train, n_test_val])

test_train_loader = DataLoader(test_train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_val_loader = DataLoader(test_val_ds, batch_size=BATCH_SIZE)
print(f'Test BS{test_bs_id} samples: {n_test_train} train, {n_test_val} val')

In [ ]:
import copy

# --- Method A: Adapt θ_BS only (freeze E + θ_task) ---
adapt_model = copy.deepcopy(pretrained_model).to(device)
adapt_model.site_embedding.reset()  # Zero-init θ_BS for new site
adapt_model.freeze_encoder()
adapt_model.freeze_task_head()

adapt_result = train_local(
    adapt_model, test_train_loader, test_val_loader,
    epochs=ADAPT_EPOCHS, lr=1e-2, device=device
)
print(f'θ_BS-only adaptation: {10*np.log10(adapt_result["best_val"]):.2f} dB')

# --- Method B: From scratch ---
scratch_model = create_model(site_integration='film', site_embed_dim=64).to(device)
scratch_result = train_local(
    scratch_model, test_train_loader, test_val_loader,
    epochs=PRETRAIN_EPOCHS, lr=1e-3, device=device
)
print(f'From scratch: {10*np.log10(scratch_result["best_val"]):.2f} dB')

# --- Method C: Fine-tune all params from pre-trained ---
finetune_model = copy.deepcopy(pretrained_model).to(device)
finetune_model.site_embedding.reset()
finetune_model.unfreeze_all()

finetune_result = train_local(
    finetune_model, test_train_loader, test_val_loader,
    epochs=ADAPT_EPOCHS, lr=1e-4, device=device  # Lower LR for fine-tuning
)
print(f'Fine-tune all: {10*np.log10(finetune_result["best_val"]):.2f} dB')

In [ ]:
# Plot: Convergence comparison
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

ax.plot(10*np.log10(adapt_result['val_losses']), label='θ_BS-only (ours)', linewidth=2)
ax.plot(10*np.log10(scratch_result['val_losses']), label='From scratch', linewidth=2)
ax.plot(10*np.log10(finetune_result['val_losses']), label='Fine-tune all', linewidth=2)

ax.set_xlabel('Epoch')
ax.set_ylabel('NMSE (dB)')
ax.set_title(f'Adaptation to Unseen BS{test_bs_id}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('step0_convergence.png', dpi=150)
plt.show()